# Explorer 03 - Formats and reissues

**Purpose.** Explore the format mix by year and settle on a definition of reissue.

**Feeds main report section:** 5.2 RQ2 - Format mix and the vinyl boom

**Modules used:** `vinyl.formats`, `vinyl.viz`

**Tables read:** `release_formats`, `releases`, `masters`, `release_genres`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-03-formats")
from vinyl import formats, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_formats = ingest.read_table(spark, snapshot, "release_formats")
releases = ingest.read_table(spark, snapshot, "releases")
masters = ingest.read_table(spark, snapshot, "masters")
release_genres = ingest.read_table(spark, snapshot, "release_genres")

## 1. What formats and descriptions exist?

In [ ]:
release_formats.groupBy("format_name").agg(F.countDistinct("release_id").alias("n")).orderBy(F.desc("n")).show(30, truncate=False)

In [ ]:
release_formats.select(F.explode("descriptions").alias("description")).groupBy("description").count().orderBy(F.desc("count")).show(40, truncate=False)

## 2. Format shares by year

In [ ]:
format_shares = formats.format_shares_by_year(release_formats).toPandas()
main_formats = ["Vinyl", "CD", "Cassette", "File"]  # TODO: confirm against the list above
viz.plot_lines(format_shares[format_shares.format_name.isin(main_formats)], "year", "share", "format_name",
               title="TODO: format shares", percent=True)

## 3. Reissue definitions

Compare the tag-based flag (`is_reissue`) with the master-based flag (`is_reissue_master`) before picking one.

In [ ]:
flagged = formats.add_master_reissue_flag(release_formats, releases, masters, min_gap_years=1)
flagged.groupBy("is_reissue", "is_reissue_master").agg(F.countDistinct("release_id").alias("n")).show()

In [ ]:
vinyl = flagged.where(F.col("format_name") == "Vinyl")
for flag in ["is_reissue", "is_reissue_master", "is_reissue_any"]:
    print(flag)
    formats.reissue_share(vinyl, by=["year"], flag_col=flag).orderBy("year").show()

## 4. Reissue share by genre

In [ ]:
by_genre = formats.reissue_share(
    flagged.join(release_genres.select("release_id", "genre"), "release_id"),
    by=["year", "format_name", "genre"],
    flag_col="is_reissue_any",
)
by_genre.where(F.col("format_name") == "Vinyl").orderBy("genre", "year").show(40)

In [ ]:
# TODO: how old is the music being reissued (year - master_year distribution)?
# TODO: which pressing plants (release_companies, "Pressed By") handle the vinyl volume?
# TODO: "File" releases - are digital releases catalogued consistently enough to compare?

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] Reissue definition used in the report (tag, master, or either) and the gap in years.
- [ ] The list of formats shown, and how minor formats are grouped.
- [ ] Share of vinyl growth explained by reissues (the headline number).